# AA3 · Examen del Parcial III · Problema 2: TrasladoSeguro
**Equipo:** (nombre o número del equipo)
**Integrantes:** (nombres)

TrasladoSeguro lleva a los pacientes que la cámara de Behe refiere a su clínica, en la ciudad de `Taxi-v3`. **En temporada de lluvias**, 8 de cada 10 movimientos salen bien y los otros 2 se desvían a un lado (`is_rainy=True`). Con `evaluar_cliente`, en 100 episodios con lluvia y sin explorar, la empresa pide:
- entregar **al menos 98 de cada 100** pacientes (`entregados_pct` ≥ 98);
- **ninguna maniobra ilegal** (`ilegales` = 0);
- **17 pasos o menos** en promedio (`pasos_promedio` ≤ 17).

**Cómo trabajar:** sigan las secciones en orden y **no cambien sus títulos**. Pueden agregar las celdas que necesiten. Al final, ejecuten todo de arriba abajo y guarden el cuaderno con sus salidas. El código que les sirve: la P9-A (Q-learning y el Taxi) y la actividad del martes 24 (SARSA).

## 1. Entorno y línea base
La celda de partida del enunciado, **sin cambios**: el entorno con lluvia, el entorno seco y la función `evaluar_cliente`. Si actualiza gymnasium, reinicien la sesión y vuelvan a correrla.

In [ ]:
import sys, subprocess
import gymnasium as gym
if tuple(int(x) for x in gym.__version__.split(".")[:2]) < (1, 2):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium>=1.2", "pygame"], check=True)
    raise SystemExit("Se actualizó gymnasium: reinicia la sesión y vuelve a correr.")
import numpy as np

lluvia = gym.make("Taxi-v3", is_rainy=True, render_mode="rgb_array")
seco = gym.make("Taxi-v3")

def evaluar_cliente(env, Q=None, n=100, semilla=2026):
    # La evaluación del cliente: sin explorar, semillas 2026 a 2125. Q=None juega al azar.
    rng = np.random.default_rng(0)
    G_lista, entregados, pasos, ilegales = [], 0, [], 0
    for ep in range(n):
        s, _ = env.reset(seed=semilla + ep)
        fin, G, k = False, 0.0, 0
        while not fin:
            a = int(rng.integers(env.action_space.n)) if Q is None else int(np.argmax(Q[s]))
            s, r, terminado, truncado, _ = env.step(a)
            G, k, fin = G + r, k + 1, terminado or truncado
            ilegales += r == -10
        G_lista.append(G); pasos.append(k)
        entregados += terminado and r == 20
    return {"recompensa_media": round(float(np.mean(G_lista)), 2), "entregados_pct": 100 * entregados / n,
            "pasos_promedio": round(float(np.mean(pasos)), 2), "ilegales": int(ilegales)}

In [ ]:
# ✏️ La línea base: el agente al azar, evaluar_cliente(lluvia)


## 2. Q-learning
La función `q_learning` del Bloque 3 de la P9-A, **sin cambios** (α = 0.1, γ = 0.99, ε de 1 a 0.05 en la primera mitad de los episodios). Entrénenla en `lluvia` con **5,000 episodios** y evalúenla con `evaluar_cliente`.

In [ ]:
# ✏️ Q-learning


## 3. SARSA
La misma función con el cambio de la actividad del martes: el objetivo usa la acción que el agente **sí hará** en `s2` (elegida con ε-greedy). Mismos hiperparámetros y **5,000 episodios**.
- Si alguno de los dos no cumple, entrénenlo otra vez con **20,000 episodios** y reporten las dos corridas.
- Grafiquen las curvas de aprendizaje en **una sola figura** (media móvil de 100 episodios).

In [ ]:
# ✏️ SARSA


In [ ]:
# ✏️ Las curvas de aprendizaje en una figura


## 4. Evaluación del cliente
- `resultados_taxi.csv`, con una fila por corrida (incluido el agente al azar) y las columnas `modelo` (`al azar`, `Q-learning` o `SARSA`), `episodios`, `recompensa_media`, `entregados_pct`, `pasos_promedio`, `ilegales`, `cumple` y `elegido` (`True` solo en el que le recomiendan al cliente).
- La tabla Q de la **última** corrida de cada algoritmo, con `np.save`, en `taxi/modelo/q_qlearning.npy` y `taxi/modelo/q_sarsa.npy` del repositorio.
- La tabla elegida evaluada **sin lluvia**: `evaluar_cliente(seco, Q)`.

In [ ]:
# ✏️ resultados_taxi.csv, las tablas Q y la evaluación sin lluvia


## 5. Un episodio con lluvia
Con la tabla elegida, un episodio completo con lluvia en cuadros (`render()`), como en el ejercicio final de la P9-A.

In [ ]:
# ✏️ Un episodio en cuadros


## 6. Respuestas
Una celda por pregunta. Usen sus números.

**R1.** ¿Qué agente le recomiendan al cliente y por qué? Usen los números de `resultados_taxi`.

*Escriban aquí su respuesta, con sus números.*

**R2.** Con la misma tabla, ¿cuántos pasos tarda sin lluvia? ¿Por qué con lluvia tarda más?

*Escriban aquí su respuesta, con sus números.*

**R3.** Si uno de los dos necesitó más episodios para cumplir, ¿por qué creen que fue? Si los dos cumplieron con 5,000, ¿cuál aprendió más rápido según las curvas?

*Escriban aquí su respuesta, con sus números.*

**R4.** En el barranco del martes 24, SARSA aprendió una ruta distinta a la de Q-learning. ¿Por qué aquí sus políticas terminan casi iguales?

*Escriban aquí su respuesta, con sus números.*

**R5.** ¿Por qué el agente no comete maniobras ilegales aunque llueva? Den una limitación de este simulador frente a una ciudad real.

*Escriban aquí su respuesta, con sus números.*